# 🚢 Titanic Survival Prediction — TensorFlow Decision Forests (Improved)

Bu notebook, Titanic yarışması için **TensorFlow Decision Forests** kullanarak:
- Gelişmiş özellik mühendisliği (Feature Engineering)
- Hiperparametre optimizasyonu
- Ensemble modelleme
- Model kaydetme (HuggingFace deploy için)

adımlarını içermektedir.

## 1. Kütüphane Yüklemeleri

In [ ]:
import numpy as np
import pandas as pd
import os
import re
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
import tensorflow_decision_forests as tfdf

print(f"TensorFlow version   : {tf.__version__}")
print(f"TF-DF version        : {tfdf.__version__}")

## 2. Veri Yükleme

In [ ]:
# Kaggle ortamında çalışıyorsa orijinal yollar; aksi hâlde yerel
if os.path.exists("/kaggle/input/titanic"):
    TRAIN_PATH   = "/kaggle/input/titanic/train.csv"
    TEST_PATH    = "/kaggle/input/titanic/test.csv"
    OUTPUT_PATH  = "/kaggle/working/"
else:
    TRAIN_PATH   = "train.csv"
    TEST_PATH    = "test.csv"
    OUTPUT_PATH  = "./"

train_df   = pd.read_csv(TRAIN_PATH)
serving_df = pd.read_csv(TEST_PATH)

print(f"Train shape   : {train_df.shape}")
print(f"Test  shape   : {serving_df.shape}")
print("\nEksik değerler (train):")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

## 3. Gelişmiş Özellik Mühendisliği

Orijinal notebook'a kıyasla eklenen iyileştirmeler:
- **Title** (unvan) çıkarımı
- **FamilySize** ve **IsAlone** özelliği
- **Deck** (güverte) çıkarımı
- **Fare** aralıklandırması (Fare_band)
- **Age** eksik değer doldurma (unvana göre medyan)
- **Embarked** eksik değer doldurma

In [ ]:
def extract_title(name: str) -> str:
    """İsimden unvanı çıkar."""
    match = re.search(r',\s*([^.]+)\.', name)
    if match:
        title = match.group(1).strip()
        # Nadir unvanları grupla
        rare = {'Lady','Countess','Capt','Col','Don','Dr',
                'Major','Rev','Sir','Jonkheer','Dona'}
        if title in rare:
            return 'Rare'
        title_map = {'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'}
        return title_map.get(title, title)
    return 'Unknown'


def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # --- Orijinal dönüşümler ---
    def normalize_name(x):
        return " ".join([v.strip(",()[].\"'") for v in x.split(" ")])

    def ticket_number(x):
        return x.split(" ")[-1]

    def ticket_item(x):
        items = x.split(" ")
        return "NONE" if len(items) == 1 else "_".join(items[:-1])

    df["Name"]          = df["Name"].apply(normalize_name)
    df["Ticket_number"] = df["Ticket"].apply(ticket_number)
    df["Ticket_item"]   = df["Ticket"].apply(ticket_item)

    # --- Yeni özellikler ---
    # Unvan
    df["Title"] = df["Name"].apply(extract_title)

    # Age — unvana göre medyan ile doldur
    age_medians = df.groupby("Title")["Age"].transform("median")
    df["Age"] = df["Age"].fillna(age_medians).fillna(df["Age"].median())

    # Aile özellikleri
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"]    = (df["FamilySize"] == 1).astype(int)

    # Güverte
    df["Deck"] = df["Cabin"].apply(
        lambda x: str(x)[0] if pd.notna(x) else "U"
    )

    # Embarked eksik → mod
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

    # Fare eksik → medyan
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())

    # Fare bandı
    df["Fare_band"] = pd.qcut(
        df["Fare"], q=4, labels=["Low","Medium","High","VHigh"]
    ).astype(str)

    # Yaş bandı
    df["Age_band"] = pd.cut(
        df["Age"],
        bins=[0, 12, 18, 35, 60, 120],
        labels=["Child","Teen","Young","Adult","Senior"]
    ).astype(str)

    return df


preprocessed_train_df   = preprocess(train_df)
preprocessed_serving_df = preprocess(serving_df)

print("Ön-işleme tamamlandı. Örnek:")
preprocessed_train_df.head(3)

## 4. Özellik Listesi

In [ ]:
# Kaldırılacak sütunlar
DROP_COLS = ["Ticket", "PassengerId", "Survived", "Cabin"]

input_features = [
    c for c in preprocessed_train_df.columns
    if c not in DROP_COLS
]

print(f"Toplam {len(input_features)} özellik:")
print(input_features)

## 5. TensorFlow Dataset Oluşturma

In [ ]:
def tokenize_names(features, labels=None):
    """İsimleri token listesine dönüştür."""
    features["Name"] = tf.strings.split(features["Name"])
    return features, labels


train_ds = (
    tfdf.keras.pd_dataframe_to_tf_dataset(
        preprocessed_train_df, label="Survived"
    ).map(tokenize_names)
)

serving_ds = (
    tfdf.keras.pd_dataframe_to_tf_dataset(preprocessed_serving_df)
    .map(tokenize_names)
)

print("Dataset'ler hazır.")

## 6. Model 1 — Varsayılan Parametreler (Baseline)

In [ ]:
baseline_model = tfdf.keras.GradientBoostedTreesModel(
    verbose=0,
    features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
    exclude_non_specified_features=True,
    random_seed=42,
)
baseline_model.fit(train_ds)

baseline_eval = baseline_model.make_inspector().evaluation()
print(f"[Baseline] Accuracy: {baseline_eval.accuracy:.4f}  Loss: {baseline_eval.loss:.4f}")

## 7. Model 2 — İyileştirilmiş Parametreler

In [ ]:
improved_model = tfdf.keras.GradientBoostedTreesModel(
    verbose=0,
    features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
    exclude_non_specified_features=True,
    min_examples=1,
    categorical_algorithm="RANDOM",
    shrinkage=0.05,
    split_axis="SPARSE_OBLIQUE",
    sparse_oblique_normalization="MIN_MAX",
    sparse_oblique_num_projections_exponent=2.0,
    num_trees=2000,
    random_seed=42,
)
improved_model.fit(train_ds)

improved_eval = improved_model.make_inspector().evaluation()
print(f"[Improved] Accuracy: {improved_eval.accuracy:.4f}  Loss: {improved_eval.loss:.4f}")

## 8. Model 3 — Hiperparametre Optimizasyonu (RandomSearch)

In [ ]:
tuner = tfdf.tuner.RandomSearch(num_trials=200)   # Kaggle'da süre varsa 500-1000
tuner.choice("min_examples", [1, 2, 5, 7, 10])
tuner.choice("categorical_algorithm", ["CART", "RANDOM"])

local_search_space = tuner.choice("growing_strategy", ["LOCAL"])
local_search_space.choice("max_depth", [3, 4, 5, 6, 8])

global_search_space = tuner.choice("growing_strategy", ["BEST_FIRST_GLOBAL"], merge=True)
global_search_space.choice("max_num_nodes", [16, 32, 64, 128, 256])

tuner.choice("shrinkage", [0.02, 0.05, 0.10, 0.15])
tuner.choice("num_candidate_attributes_ratio", [0.2, 0.5, 0.9, 1.0])

oblique_space = tuner.choice("split_axis", ["SPARSE_OBLIQUE"], merge=True)
oblique_space.choice("sparse_oblique_normalization", ["NONE", "STANDARD_DEVIATION", "MIN_MAX"])
oblique_space.choice("sparse_oblique_weights", ["BINARY", "CONTINUOUS"])
oblique_space.choice("sparse_oblique_num_projections_exponent", [1.0, 1.5])

tuned_model = tfdf.keras.GradientBoostedTreesModel(
    verbose=0,
    features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
    exclude_non_specified_features=True,
    tuner=tuner,
    random_seed=42,
)
tuned_model.fit(train_ds, verbose=0)

tuned_eval = tuned_model.make_inspector().evaluation()
print(f"[Tuned]    Accuracy: {tuned_eval.accuracy:.4f}  Loss: {tuned_eval.loss:.4f}")

## 9. Model 4 — Ensemble (100 Farklı Seed)

In [ ]:
predictions    = None
num_predictions = 0

for i in range(100):
    model_i = tfdf.keras.GradientBoostedTreesModel(
        verbose=0,
        features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
        exclude_non_specified_features=True,
        min_examples=1,
        categorical_algorithm="RANDOM",
        shrinkage=0.05,
        split_axis="SPARSE_OBLIQUE",
        sparse_oblique_normalization="MIN_MAX",
        sparse_oblique_num_projections_exponent=2.0,
        num_trees=500,   # Ensemble için daha az ağaç yeterli
        random_seed=i,
        honest=True,
    )
    model_i.fit(train_ds)

    sub_preds = model_i.predict(serving_ds, verbose=0)[:, 0]
    predictions = sub_preds if predictions is None else predictions + sub_preds
    num_predictions += 1

ensemble_probs = predictions / num_predictions
print("Ensemble tamamlandı.")

## 10. Tahmin & Submission

In [ ]:
def prediction_to_kaggle_format(proba, threshold=0.5):
    return pd.DataFrame({
        "PassengerId": serving_df["PassengerId"],
        "Survived": (proba >= threshold).astype(int)
    })

def make_submission(kaggle_predictions, filename="submission.csv"):
    path = os.path.join(OUTPUT_PATH, filename)
    kaggle_predictions.to_csv(path, index=False)
    print(f"Submission kaydedildi: {path}")
    return path

# Ensemble submission
ensemble_submission = prediction_to_kaggle_format(ensemble_probs)
make_submission(ensemble_submission, "submission_ensemble.csv")
ensemble_submission.head()

## 11. Model Karşılaştırma Özeti

In [ ]:
results = pd.DataFrame([
    {"Model": "Baseline GBT",           "Accuracy": baseline_eval.accuracy, "Loss": baseline_eval.loss},
    {"Model": "Improved GBT",           "Accuracy": improved_eval.accuracy, "Loss": improved_eval.loss},
    {"Model": "Tuned GBT (RandomSearch)","Accuracy": tuned_eval.accuracy,   "Loss": tuned_eval.loss},
])
print(results.to_string(index=False))

## 12. En İyi Modeli Kaydet (HuggingFace için)

In [ ]:
import json

MODEL_SAVE_PATH = os.path.join(OUTPUT_PATH, "titanic_gbt_model")

# En yüksek accuracy'e sahip modeli seç
best_model = max(
    [(baseline_model, baseline_eval.accuracy),
     (improved_model, improved_eval.accuracy),
     (tuned_model,    tuned_eval.accuracy)],
    key=lambda x: x[1]
)[0]

best_model.save(MODEL_SAVE_PATH)
print(f"Model kaydedildi: {MODEL_SAVE_PATH}")

# Özellik meta-verisi kaydet (inference için)
meta = {
    "input_features": input_features,
    "label": "Survived",
    "drop_cols": DROP_COLS
}
with open(os.path.join(OUTPUT_PATH, "model_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)
print("Meta veri kaydedildi: model_meta.json")

## 13. Değişken Önem Analizi

In [ ]:
inspector = best_model.make_inspector()

print("=== Değişken Önemi (NUM_AS_ROOT) ===")
for importance in inspector.variable_importances().get("NUM_AS_ROOT", [])[:10]:
    print(f"  {importance.name.name:<25} {importance.importance:.4f}")